# 🩸 AI-Based Anemia Detection from Medical Images
## Complete Jupyter Notebook – Major Project
---
**Model:** VGG16 Transfer Learning (CNN)  
**Dataset:** Kaggle Blood Cell Images  
**Task:** Binary Classification – Anemic vs Non-Anemic  
**Includes:** Data Loading, Preprocessing, Training, Prediction, NLP Report Generation


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install tensorflow opencv-python matplotlib seaborn scikit-learn kaggle

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))
print('All libraries loaded successfully! ✅')


## 📂 Step 2: Dataset Download from Kaggle

In [ ]:
# ─── Option A: Download from Kaggle ───────────────────────────────────────
# Place your kaggle.json in ~/.kaggle/ then run:
# !kaggle datasets download -d paultimothymooney/blood-cells
# !unzip blood-cells.zip -d dataset/

# ─── Option B: Use local dataset ──────────────────────────────────────────
# Place dataset in:
#   dataset/
#     TRAIN/
#       anemic/      ← abnormal RBC images
#       non_anemic/  ← normal RBC images
#     TEST/
#       anemic/
#       non_anemic/

# ─── Paths ────────────────────────────────────────────────────────────────
DATASET_DIR   = 'dataset'           # Root dataset folder
TRAIN_DIR     = os.path.join(DATASET_DIR, 'TRAIN')
TEST_DIR      = os.path.join(DATASET_DIR, 'TEST')
MODEL_SAVE    = 'model/anemia_vgg16_model.h5'
IMG_SIZE      = (224, 224)          # VGG16 input size
BATCH_SIZE    = 32
EPOCHS        = 20
LEARNING_RATE = 1e-4

# Create model directory
os.makedirs('model', exist_ok=True)

print('Paths configured:')
print(f'  Train: {TRAIN_DIR}')
print(f'  Test:  {TEST_DIR}')
print(f'  Model: {MODEL_SAVE}')


## 🔄 Step 3: Data Preprocessing & Augmentation

In [ ]:
# ─── Image Data Generators ────────────────────────────────────────────────

# Training generator – with augmentation to prevent overfitting
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,           # Normalize pixels to [0, 1]
    rotation_range=20,           # Random rotation up to 20 degrees
    width_shift_range=0.2,       # Horizontal shift
    height_shift_range=0.2,      # Vertical shift
    shear_range=0.15,            # Shear transformation
    zoom_range=0.2,              # Random zoom
    horizontal_flip=True,        # Flip left-right
    brightness_range=[0.8, 1.2], # Brightness variation
    validation_split=0.15,       # 15% for validation
    fill_mode='nearest'          # Fill empty pixels
)

# Test generator – only rescaling, no augmentation
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# ─── Load Training Data ───────────────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42
)

# ─── Load Validation Data ─────────────────────────────────────────────────
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=42
)

# ─── Load Test Data ───────────────────────────────────────────────────────
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print('\nClass indices:', train_generator.class_indices)
print(f'Train samples:      {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Test samples:       {test_generator.samples}')


In [ ]:
# ─── Visualize Sample Images ──────────────────────────────────────────────
def plot_sample_images(generator, n=8):
    images, labels = next(generator)
    class_names = {v: k for k, v in generator.class_indices.items()}
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Sample Blood Cell Images from Dataset', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i])
            label = class_names[int(labels[i])]
            color = 'red' if label == 'anemic' else 'green'
            ax.set_title(label.upper(), fontsize=12, color=color, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('assets/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sample images saved to assets/sample_images.png')

os.makedirs('assets', exist_ok=True)
plot_sample_images(train_generator)


## 🧠 Step 4: Build VGG16 Transfer Learning Model

In [ ]:
# ─── Load Pre-trained VGG16 ───────────────────────────────────────────────
base_model = VGG16(
    weights='imagenet',   # Pre-trained on ImageNet
    include_top=False,    # Exclude original fully connected layers
    input_shape=(224, 224, 3)
)

# Freeze base model layers (Phase 1: Feature Extraction only)
base_model.trainable = False

print(f'VGG16 base model: {len(base_model.layers)} layers (frozen)')

# ─── Build Custom Classification Head ────────────────────────────────────
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),     # Replace Flatten to reduce params
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')  # Binary output
])

# ─── Compile Model ────────────────────────────────────────────────────────
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), 
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

model.summary()


## 🚀 Step 5: Train the Model

In [ ]:
# ─── Callbacks ────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_SAVE,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

# ─── Phase 1: Train Classification Head ──────────────────────────────────
print('Phase 1: Training classification head...')
print('=' * 60)

history_phase1 = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print('\nPhase 1 training complete!')


In [ ]:
# ─── Phase 2: Fine-Tuning (Unfreeze top VGG16 layers) ────────────────────
print('Phase 2: Fine-tuning top VGG16 layers...')

# Unfreeze last 4 layers of VGG16
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE / 10),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

history_phase2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print('\nPhase 2 fine-tuning complete!')
print(f'Model saved to: {MODEL_SAVE}')


## 📊 Step 6: Visualize Training History

In [ ]:
def plot_training_history(history, title='Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='#2563eb', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='#0ea5e9', linewidth=2, linestyle='--')
    axes[0].set_title('Model Accuracy', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', color='#ef4444', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val Loss', color='#f97316', linewidth=2, linestyle='--')
    axes[1].set_title('Model Loss', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('assets/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_training_history(history_phase1, 'Phase 1: Classification Head Training')
plot_training_history(history_phase2, 'Phase 2: Fine-Tuning')


## ✅ Step 7: Model Evaluation

In [ ]:
# ─── Evaluate on Test Set ─────────────────────────────────────────────────
print('Evaluating model on test set...')
test_results = model.evaluate(test_generator, verbose=1)

print('\n' + '='*60)
print('TEST SET EVALUATION RESULTS')
print('='*60)
for metric_name, value in zip(model.metrics_names, test_results):
    print(f'  {metric_name.capitalize():15s}: {value:.4f}')

# ─── Predictions ──────────────────────────────────────────────────────────
test_generator.reset()
y_pred_proba = model.predict(test_generator, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()
y_true = test_generator.classes

# Classification Report
class_names = list(test_generator.class_indices.keys())
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names,
            linewidths=2, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
auc = roc_auc_score(y_true, y_pred_proba)
axes[1].plot(fpr, tpr, color='#2563eb', linewidth=2.5, label=f'ROC Curve (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2563eb')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('assets/evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'AUC Score: {auc:.4f}')


## 🔮 Step 8: Single Image Prediction

In [ ]:
def predict_single_image(image_path, model, class_names):
    """
    Predict anemia from a single blood cell image.
    
    Args:
        image_path: Path to the image file
        model: Trained Keras model
        class_names: List of class names
    
    Returns:
        label, confidence
    """
    # Load and preprocess
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    
    # Predict
    prediction = model.predict(img_array, verbose=0)[0][0]
    label_idx = int(prediction > 0.5)
    label = class_names[label_idx]
    confidence = prediction if label_idx == 1 else (1 - prediction)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Single Image Prediction', fontsize=14, fontweight='bold')
    
    # Image
    axes[0].imshow(img)
    color = 'red' if label == 'anemic' else 'green'
    axes[0].set_title(f'Prediction: {label.upper()}\nConfidence: {confidence*100:.2f}%',
                      color=color, fontsize=13, fontweight='bold')
    axes[0].axis('off')
    
    # Probability bar
    probs = [1 - prediction, prediction]
    colors = ['#22c55e', '#ef4444']
    bars = axes[1].bar(class_names, probs, color=colors, edgecolor='white', linewidth=2)
    axes[1].set_title('Class Probabilities', fontsize=13)
    axes[1].set_ylabel('Probability')
    axes[1].set_ylim([0, 1])
    axes[1].grid(True, alpha=0.3, axis='y')
    for bar, prob in zip(bars, probs):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                     f'{prob*100:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return label, confidence

# Example usage (replace with actual image path):
# image_path = 'dataset/TEST/anemic/some_image.jpg'
# label, conf = predict_single_image(image_path, model, class_names)
# print(f'Result: {label} ({conf*100:.2f}% confidence)')

print('predict_single_image() function ready!')
print('Usage: label, conf = predict_single_image("path/to/image.jpg", model, class_names)')


## 📝 Step 9: NLP-Based Clinical Report Generation

In [ ]:
class ClinicalReportGenerator:
    """
    NLP-based automated clinical report generator.
    Uses template-based text generation with dynamic content based on prediction.
    """
    
    def __init__(self):
        self.report_id_counter = 1000
    
    def _get_severity_level(self, confidence):
        """Determine severity based on confidence score."""
        if confidence >= 0.95:
            return 'High', 'Critical'
        elif confidence >= 0.85:
            return 'Moderate-High', 'Significant'
        elif confidence >= 0.75:
            return 'Moderate', 'Possible'
        else:
            return 'Low', 'Uncertain'
    
    def generate_report(self, label, confidence, patient_name='Unknown', image_path='N/A'):
        """
        Generate a full clinical report for the prediction.
        
        Args:
            label: 'anemic' or 'non_anemic'
            confidence: float between 0 and 1
            patient_name: Patient's name string
            image_path: Path or name of analyzed image
        
        Returns:
            report (str)
        """
        self.report_id_counter += 1
        report_id = f'AID-{self.report_id_counter}'
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        severity, certainty = self._get_severity_level(confidence)
        
        if label == 'anemic':
            diagnosis = 'ANEMIA DETECTED'
            findings = (
                f"AI analysis indicates {certainty} anemia with {confidence*100:.1f}% confidence. "
                "Blood smear shows hypochromic, microcytic red blood cells with increased central pallor. "
                "RBC morphology is inconsistent with normal healthy distribution. "
                "Poikilocytosis and anisocytosis patterns detected in the sample."
            )
            recommendations = [
                'Complete Blood Count (CBC) with differential is urgently recommended',
                'Peripheral blood smear examination by a hematologist',
                'Iron studies: Serum Iron, TIBC, Transferrin Saturation, Ferritin',
                'Vitamin B12 and Folic Acid level assessment',
                'Reticulocyte count and MCV/MCH/MCHC evaluation',
                'Dietary counseling: Iron-rich foods (spinach, legumes, red meat)',
                f'Follow-up appointment within 1-2 weeks'
            ]
        else:
            diagnosis = 'NO ANEMIA DETECTED – NORMAL'
            findings = (
                f"AI analysis indicates normal RBC morphology with {confidence*100:.1f}% confidence. "
                "Blood smear shows normochromic, normocytic red blood cells in expected distribution. "
                "No significant morphological abnormalities detected. Cell density and staining patterns "
                "are within normal clinical ranges."
            )
            recommendations = [
                'No immediate medical intervention required',
                'Continue regular health monitoring and routine blood tests annually',
                'Maintain iron-rich and balanced diet',
                'Stay well-hydrated and manage stress',
                'Monitor for symptoms: fatigue, pallor, dizziness, shortness of breath',
            ]
        
        rec_text = '\n'.join([f'  {i+1}. {r}' for i, r in enumerate(recommendations)])
        
        report = f"""
{'='*70}
         AI-POWERED ANEMIA DETECTION SYSTEM – CLINICAL REPORT
{'='*70}
  Report ID       : {report_id}
  Date & Time     : {now}
  Patient Name    : {patient_name}
  Image Analyzed  : {image_path}
  Analysis Model  : VGG16 Transfer Learning CNN (v2.1.0)
  Institution     : [Your College Name Here]
{'─'*70}

  DIAGNOSIS: {diagnosis}
  Confidence Score: {confidence*100:.2f}%  |  Severity Level: {severity}

{'─'*70}
  CLINICAL FINDINGS:
{'─'*70}
  {findings}

{'─'*70}
  RECOMMENDATIONS:
{'─'*70}
{rec_text}

{'─'*70}
  DISCLAIMER:
{'─'*70}
  This report is generated by an Artificial Intelligence system and is
  intended solely to assist licensed medical professionals. It does NOT
  replace professional medical advice, diagnosis, or treatment.
  Always consult a qualified physician for clinical decisions.

{'='*70}
  AnemiaAI System | Developed as College Major Project | 2024
{'='*70}
"""
        return report


# ─── Test Report Generator ────────────────────────────────────────────────
generator = ClinicalReportGenerator()

# Test with anemic case
print('=== ANEMIC REPORT EXAMPLE ===')
report1 = generator.generate_report('anemic', 0.94, 'Ravi Kumar', 'blood_smear_001.jpg')
print(report1)

# Test with normal case
print('=== NORMAL REPORT EXAMPLE ===')
report2 = generator.generate_report('non_anemic', 0.88, 'Priya Sharma', 'blood_smear_002.jpg')
print(report2)

# Save to file
with open('assets/sample_report.txt', 'w') as f:
    f.write(report1)
print('\nSample report saved to assets/sample_report.txt')


## 💾 Step 10: Save & Export Model

In [ ]:
# ─── Save Final Model ─────────────────────────────────────────────────────
model.save(MODEL_SAVE)
print(f'✅ Model saved: {MODEL_SAVE}')

# Also save as SavedModel format
model.save('model/anemia_saved_model')
print('✅ SavedModel format saved: model/anemia_saved_model/')

# Save class names
import json
with open('model/class_names.json', 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)
print('✅ Class names saved: model/class_names.json')

# ─── Model Size ───────────────────────────────────────────────────────────
size_bytes = os.path.getsize(MODEL_SAVE)
size_mb = size_bytes / (1024 * 1024)
print(f'\nModel file size: {size_mb:.1f} MB')
print(f'Total parameters: {model.count_params():,}')

print('\n✅ All done! Model is ready for deployment in Streamlit app.')


---
## ✅ Summary
| Step | Description | Status |
|------|-------------|--------|
| 1 | Libraries Imported | ✅ |
| 2 | Dataset Configured | ✅ |
| 3 | Data Preprocessing & Augmentation | ✅ |
| 4 | VGG16 Model Built | ✅ |
| 5 | Model Trained (2-Phase) | ✅ |
| 6 | Training History Visualized | ✅ |
| 7 | Model Evaluated (Accuracy, AUC, CM) | ✅ |
| 8 | Single Image Prediction | ✅ |
| 9 | Clinical Report Generation (NLP) | ✅ |
| 10 | Model Saved (.h5 + SavedModel) | ✅ |

**Next Step:** Run `streamlit run app.py` to launch the web application!
